# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import gc #noqa
import sys
from pathlib import Path

import mlflow
import torch
from mlflow import MlflowClient
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

MLFLOW_DB = PROJECT_ROOT / "logs" / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"Tracking URI : sqlite:///{MLFLOW_DB}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
PROJECT_ROOT : c:\nlp_template_decoder
Tracking URI : sqlite:///c:\nlp_template_decoder\logs\mlflow.db


# Load Models & Merge

In [11]:
MODEL_NAME  = "tiny-random-LlamaForCausalLM_LoRA"
BASE_MODEL  = "HuggingFaceM4/tiny-random-LlamaForCausalLM"
OUTPUT_PATH = PROJECT_ROOT / "models" / "merged"

# Последняя версия по номеру
client = MlflowClient()
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest = max(versions, key=lambda v: int(v.version))
RUN_ID = latest.run_id
print(f"Используем: {MODEL_NAME} v{latest.version}  run_id={RUN_ID}")

# Скачать адаптер
print("Скачивание адаптера из MLflow...")
lora_path = mlflow.artifacts.download_artifacts(
    run_id=RUN_ID,
    artifact_path="lora_weights/peft",
)
print(f"Адаптер: {lora_path}")

# Базовая модель
print("Загрузка базовой модели...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu",
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Слияние
print("Навешивание LoRA и слияние...")
merged = PeftModel.from_pretrained(base_model, lora_path).merge_and_unload()

if getattr(merged.generation_config, "pad_token_id", None) in (None, -1):
    merged.generation_config.pad_token_id = tokenizer.eos_token_id

# Сохранение
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
print(f"Сохранение в {OUTPUT_PATH}...")
merged.save_pretrained(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)

del merged, base_model
gc.collect()
print("Готово!")

Используем: tiny-random-LlamaForCausalLM_LoRA v3  run_id=5259379624d5438e8fc1f87310af40bc
Скачивание адаптера из MLflow...


Адаптер: C:\Users\2BA0~1\AppData\Local\Temp\tmpyxcl14qj\lora_weights\peft
Загрузка базовой модели...



[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got -1. This may result in unexpected behavior.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 21/21 [00:00<00:00, 21016.56it/s]


Навешивание LoRA и слияние...
Сохранение в c:\nlp_template_decoder\models\merged...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 99.98it/s]


Готово!
